## Linear Regression

#### 线性回归用来做什么？
线性回归是通过一些变量的值来预测一个变量，被预测的变量叫做因变量

在这个notebook里，我们以房产价格预测为例

#### 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hvplot.pandas

from sklearn.model_selection import train_test_split

from sklearn import metrics

from sklearn.linear_model import LinearRegression

%matplotlib inline

In [ ]:
df=pd.read_csv('./dataset/Real estate.csv')


#### Check out the data
before training the model, it is necessary to know what the data looks like!

In [ ]:
# 这里我们只是一个简单的例子，所以对于date列我们不做处理，直接用float表示
df.head()



In [ ]:
df.shape


In [ ]:
df.info()


##### Correlation
现在我们用correlation看下，哪些变量对于 单位房价 - Y 的影响最大

In [ ]:
df.corr()


In [ ]:
sns.heatmap(df.corr(), annot=True,cmap='Reds')

#### Exploratory Data Analysis - EDA 来研究一下不同变量之间的关系

In [ ]:
# sns.pairplot(df)
sns.pairplot(df.loc[:, df.columns != "No"])



In [ ]:
# 有人会想，为什么对角线上是直方图呢？因为sns.pairplot默认对数值型变量绘制直方图，对类别型变量绘制条形图。下面我们直接plot出x3的直方图做一个对比。
sns.histplot(df["X3 distance to the nearest MRT station"], bins=30, kde=True)
plt.title("Distribution of X3 distance to the nearest MRT station")
plt.show()


### Train a model
下面我们开始训练一个简单的模型

#### 准备数据 Train Test Split

In [ ]:
X=df.drop('Y house price of unit area', axis=1)

y=df['Y house price of unit area']
print("X=",X.shape,"\ny=", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=666)

#### Linear Regression Model
这里是直接用的sklearn的LinearRegression 模型

In [ ]:
# from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)


In [ ]:
model.coef_

In [ ]:
pd.DataFrame(model.coef_, X.columns, columns=['Coedicients'])
X.shape

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
y_pred

#### 常用的 Evaluation Metrics
这里我们介绍几种常用的回归评估指标（Evaluation Metrics）：

- **Mean Absolute Error (MAE)**  
  表示预测值与真实值之间绝对误差的平均值：
  $$
  \mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n} \left| y_i - \hat{y}_i \right|
  $$

- **Mean Squared Error (MSE)**  
  表示预测误差平方的平均值，对较大的误差更加敏感：
  $$
  \mathrm{MSE} = \frac{1}{n}\sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2
  $$

- **Root Mean Squared Error (RMSE)**  
  为均方误差的平方根，与原始目标变量具有相同的量纲：
  $$
  \mathrm{RMSE} = \sqrt{ \frac{1}{n}\sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2 }
  $$

In [ ]:
from sklearn import metrics
MAE= metrics.mean_absolute_error(y_test, y_pred)
MSE=metrics.mean_squared_error(y_test, y_pred)
RMSE= np.sqrt(MSE)

### Residual Histogram 残差直方图
Residual Histogram 是一个非常直观又非常有意思的vis， 可以作为evaluation metric的一个补充，下面我们来一起看一下。
那首先，残差 这里我们是 y_test - y_pred, 作为横轴
而 density则是作为y轴


In [ ]:
test_residual= y_test - y_pred
pd.DataFrame({'Error Values': (test_residual)}).hvplot.kde()


In [ ]:
sns.displot(test_residual, bins=25, kde=True)


In [ ]:
sns.scatterplot(x=y_test, y=test_residual)
plt.axhline(y=0, color='r', ls='--')

In [ ]:
plt.scatter(y_test, y_pred)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)
plt.xlabel("y_test")
plt.ylabel("y_pred")
plt.title("Prediction vs Ground Truth")
plt.show()


## 问题
好的，那现在你知道linear regression 是怎么回事了，这里有一个开放性问题，所谓的“拟合” 到底是怎么拟合的？
还记得前面我们是怎么获得模型的吗？
```python
# from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
```
那么这串代码是怎么算出的参数呢？
最小二乘法 ordinary least squares.
数学就是这么丰富的被用在计算机里，当时学CS时，还是对数学的理解太浅了！！！